### Install Library yang digunakan

Uncomment cell berikut kemudian run 

In [1]:
# ! pip install selenium bs4 pandas tqdm

### Import library

In [2]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from bs4 import BeautifulSoup
from tqdm import tqdm
import urllib.parse
import pandas as pd
import re
import time

### Function Utility

In [3]:
def extract_lat_lng(url):
    match = re.search(r'/@(-?\d+\.\d+),(-?\d+\.\d+)', url)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None


def wait_for_url_coordinates(driver, max_attempts=10):
    for _ in range(max_attempts):
        url = driver.current_url
        if '/@' in url and re.search(r'/@-?\d+\.\d+,-?\d+\.\d+', url):
            return url
        time.sleep(1)
    return driver.current_url


def wait_page_change(driver, timeout=10):
    initial = driver.page_source
    wait = WebDriverWait(driver, timeout)
    wait.until(lambda d: d.page_source != initial)


def get_google_name(soup):
    h1 = soup.find('h1', class_='DUwDvf lfPIob')
    return h1.get_text(strip=True) if h1 else None

### Main Scrapper

In [4]:
def get_long_lat(driver, location, nmdesa):

    search_query = f"{location} {nmdesa}"
    encoded = urllib.parse.quote(search_query)

    driver.get(f"https://www.google.com/maps/search/{encoded}")
    driver.implicitly_wait(3)

    wait_page_change(driver)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # CASE 1: Multiple results
    if soup.find('h1', class_="fontTitleLarge IFMGgb"):

        link = soup.find('a', class_='hfpxzc')
        if not link:
            return [None, None, None, None, 1]

        driver.get(link.get("href"))
        wait_page_change(driver)

        url = wait_for_url_coordinates(driver)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        nama_google = get_google_name(soup)

        lat, lng = extract_lat_lng(url)

        if lat and lng:
            return [lat, lng, nama_google, url, 0]
        return [None, None, nama_google, url, 1]

    # CASE 2: Direct result
    if soup.find('h1', class_='DUwDvf lfPIob'):

        url = wait_for_url_coordinates(driver)
        nama_google = get_google_name(soup)

        lat, lng = extract_lat_lng(url)

        if lat and lng:
            return [lat, lng, nama_google, url, 0]
        return [None, None, nama_google, url, 1]

    # CASE 3: Not found
    return [None, None, None, None, 2]

## DATA PREPARATION

In [5]:
file_input = "Data-SBR.csv"

df = pd.read_csv(f"proses/{file_input}", sep=";", dtype=object)

for col in ["latitude", "longitude", "nama_google", "url", "isna"]:
    if col not in df.columns:
        df[col] = ""


df_na = df[df["latitude"].isna() | (df["latitude"] == "")]
df_no_na = df[~(df["latitude"].isna() | (df["latitude"] == ""))]

df_na["isna"] = df_na["isna"].apply(lambda x: 1 if (pd.isna(x) or x != 2) else 2)
df_no_na["isna"] = 0

C:\Users\Fawcet Jenusdy Makay\AppData\Local\Temp\ipykernel_20540\3032876942.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_na["isna"] = df_na["isna"].apply(lambda x: 1 if (pd.isna(x) or x != 2) else 2)
C:\Users\Fawcet Jenusdy Makay\AppData\Local\Temp\ipykernel_20540\3032876942.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_na["isna"] = 0


In [6]:
df

,idsbr,nama,alamat,kdprov,kdkab,kdkec,kddesa,nmprov,nmkab,nmkec,nmdesa,status,latitude,longitude,nama_google,url,isna
0,60145808,SOTO SUKOHARJO DAN KOPI KLOTHOK,JALAN BEKASI TIMUR IV,31,72,060,005,DKI JAKARTA,JAKARTA TIMUR,JATINEGARA,CIPINANG BESAR UTARA,Aktif,NaN,NaN,NaN,NaN,2
1,60145898,NOW CUT BARBERSHOP,JALAN BEKASI TIMUR IV,31,72,060,005,DKI JAKARTA,JAKARTA TIMUR,JATINEGARA,CIPINANG BESAR UTARA,Aktif,-6.21725,106.8758626,Nowcut Barbershop,https://www.google.com/maps/place/Nowcut+Barbe...,0
2,60146058,KEBAB BINTANG,JALAN BEKASI TIMUR IV,31,72,060,005,DKI JAKARTA,JAKARTA TIMUR,JATINEGARA,CIPINANG BESAR UTARA,Aktif,NaN,NaN,NaN,NaN,2


### Selenium Driver

In [7]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-extensions")

driver = webdriver.Chrome(options=options)

### Scrapping Loop

In [8]:
for i, r in tqdm(df_na.iterrows(), total=len(df_na)):

    if r["isna"] != 1:
        continue

    location = r["nama"]
    desa = r["nmdesa"]

    result = get_long_lat(driver, location, desa)

    df_na.loc[i, ["latitude", "longitude", "nama_google", "url", "isna"]] = result

driver.quit()

100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:06<00:00,  3.20s/it]


## Merge Result

In [9]:
result_df = pd.concat([df_na, df_no_na]).sort_index()

result_df.to_csv(f'proses/{file_input}', index=False, sep=';')

print(f"Total records: {len(result_df)}")
print(f"Records with coordinates: {len(result_df[result_df['latitude'] != ''])}")
print(f"Records without coordinates: {len(result_df[result_df['latitude'] == ''])}")

Total records: 3
Records with coordinates: 3
Records without coordinates: 0
